### Build A Basic ChatBot with LangGraph(Graph API)

# 🧠 LangGraph — Plain English Sketch

## Core Components

### 1️⃣ Node = A Worker
A node is a function that does one job.

Examples:
- Convert YouTube video → Transcript
- Generate Title from Transcript
- Generate Content from Title + Transcript

Think of it like:
> A factory worker that performs one task.

---

### 2️⃣ Edge = Conveyor Belt
An edge connects nodes.

It tells the system:
> "After this node finishes, go to that node."

Example flow:

START → Node A → Node B → Node C → END

Edges = the arrows between nodes.

---

### 3️⃣ State = Shared Notebook (Memory)

State is a dictionary that travels between nodes.

Each node:
- Reads from state
- Adds something to state
- Passes it forward

Example evolution:

Initial State:
{
  "url": "youtube.com/..."
}

After Transcript Node:
{
  "url": "...",
  "transcript": "..."
}

After Title Node:
{
  "url": "...",
  "transcript": "...",
  "title": "..."
}

After Content Node:
{
  "url": "...",
  "transcript": "...",
  "title": "...",
  "content": "..."
}

State grows step-by-step.

---

# 🏗 Basic Graph Structure

START → NODE → END

More complex:

START → Think Node → Tool Node → Reflect Node → END

---

# 🔥 What LangGraph Really Is

LangGraph lets you:
- Control reasoning steps
- Chain multiple AI calls
- Add tools
- Add reflection loops
- Build multi-agent systems

Without LangGraph:
One LLM call.

With LangGraph:
A structured reasoning system.

---

# 🧩 One-Line Summary

Node = Brain  
Edge = Arrow  
State = Memory  
LangGraph = Structured multi-step AI system

In [16]:
import langgraph as lg
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages




In [17]:
class State(TypedDict):
    # Masseges have the type "List". The add_messages function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends new messages to the list, rather than overwritting them)
    messages: Annotated[list, add_messages]

graph_builder=StateGraph(State)

In [18]:
graph_builder

In [19]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [27]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [28]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024E8D6A0E90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024E8D99C640>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [29]:
def chatbot(state:State):
    return {"messages" : [llm.invoke(state["messages"])]}


In [30]:
graph_builder=StateGraph(State)
## Adding nodes
graph_builder.add_node("llmchatbot",chatbot)
## Adding edges
graph_builder.add_edge(START,"llmchatbot")
graph_builder.add_edge("llmchatbot",END)

## compile the graph to check for errors
graph=graph_builder.compile()


In [36]:
response=graph.invoke({"messages": "Hello, my name is Asfand Yar."})

In [38]:
response["messages"][-1].content

"Hello Asfand Yar, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [42]:
for event in graph.stream({"messages": "Hello, my name is Asfand Yar."}):
    for value in event.values():
        print(value)

{'messages': [AIMessage(content="Hello Asfand Yar, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 45, 'total_tokens': 74, 'completion_time': 0.049051949, 'completion_tokens_details': None, 'prompt_time': 0.012372231, 'prompt_tokens_details': None, 'queue_time': 0.18217099, 'total_time': 0.06142418}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ca59d-e607-7db0-a10b-e32e4292287d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 29, 'total_tokens': 74})]}
